<a href="https://colab.research.google.com/github/MeouchyCharbel/Neural-Machine-Translation/blob/feature%2Fevaluation-improvements/04_external_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip install -q transformers datasets torch sentencepiece sacremoses sacrebleu pandas


In [2]:

import gc
import time
import torch
import pandas as pd
import sacrebleu

from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive, files

drive.mount("/content/drive")

MODEL_NAME = "Helsinki-NLP/opus-mt-en-fr"

FINETUNED_PATH = (
    "/content/drive/MyDrive/"
    "NMT_Finetuning/extended/checkpoint-1250"
)

assert Path(FINETUNED_PATH).is_dir()
assert torch.cuda.is_available()

device = "cuda"
batch_size = 8

print("Loading FLORES-200 external dataset...")

external_dataset = load_dataset(
    "yash9439/flores200",
    split="devtest",
    revision="3c6628a4571f383d029d6e897a89ac953ae756d3"
)

external_df = external_dataset.to_pandas()[
    ["eng_Latn", "fra_Latn"]
].copy()

external_df.columns = ["en", "fr"]

assert len(external_df) == 1012
assert external_df["en"].notna().all()
assert external_df["fr"].notna().all()

english_sentences = external_df["en"].tolist()
french_references = external_df["fr"].tolist()

print("External dataset:", "FLORES-200 devtest")
print("Total sentences:", len(external_df))
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def generate_translations(current_model, sentences):
    predictions = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.inference_mode():
            generated = current_model.generate(
                **inputs,
                num_beams=1,
                do_sample=False,
                max_length=256
            )

        translations = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        predictions.extend(translations)

    return predictions

models = {
    "Original Pretrained": MODEL_NAME,
    "Extended Fine-tuned": FINETUNED_PATH
}

results = []

for name, source in models.items():
    print("\nEvaluating:", name)

    current_model = AutoModelForSeq2SeqLM.from_pretrained(
        source
    ).to(device)

    current_model.eval()

    generate_translations(
        current_model,
        english_sentences[:batch_size]
    )

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    predictions = generate_translations(
        current_model,
        english_sentences
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    assert len(predictions) == 1012

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [french_references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [french_references]
    ).score

    column = (
        "original_translation"
        if name == "Original Pretrained"
        else "finetuned_translation"
    )

    external_df[column] = predictions

    results.append({
        "Model": name,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2),
        "GPU Time (s)": round(elapsed, 2)
    })

    print("BLEU:", round(bleu, 2))
    print("chrF:", round(chrf, 2))

    del current_model
    gc.collect()
    torch.cuda.empty_cache()

external_metrics_df = pd.DataFrame(results)

print("\nEXTERNAL BENCHMARK RESULTS")
display(external_metrics_df)

bleu_improvement = (
    external_metrics_df.iloc[1]["BLEU"]
    - external_metrics_df.iloc[0]["BLEU"]
)

chrf_improvement = (
    external_metrics_df.iloc[1]["chrF"]
    - external_metrics_df.iloc[0]["chrF"]
)

print("\nFINE-TUNING DIFFERENCE")
print("BLEU difference:", round(bleu_improvement, 2))
print("chrF difference:", round(chrf_improvement, 2))

external_metrics_df.to_csv(
    "external_flores_metrics.csv",
    index=False
)

external_df.to_csv(
    "external_flores_predictions.csv",
    index=False
)

print("\nExternal evaluation completed.")
print("Results saved successfully.")


Mounted at /content/drive
Loading FLORES-200 external dataset...


README.md:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

dev.parquet: reconstructing file:   0%|          |  0.00B / 21.5MB            

dev.parquet: downloading bytes:           |  0.00B            

devtest.parquet: reconstructing file:   0%|          |  0.00B / 22.5MB            

devtest.parquet: downloading bytes:           |  0.00B            

... (16 outputs left out)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

BLEU: 46.86
chrF: 69.23

EXTERNAL BENCHMARK RESULTS


,Model,BLEU,chrF,GPU Time (s)
0,Original Pretrained,46.51,69.00,60.20
1,Extended Fine-tuned,46.86,69.23,55.52



FINE-TUNING DIFFERENCE
BLEU difference: 0.35
chrF difference: 0.23

External evaluation completed.
Results saved successfully.


In [3]:

from google.colab import files

assert len(external_df) == 1012
assert len(external_metrics_df) == 2

files.download("external_flores_metrics.csv")
files.download("external_flores_predictions.csv")

print("EXTERNAL EVALUATION RESULTS EXPORTED")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

EXTERNAL EVALUATION RESULTS EXPORTED


In [4]:

import numpy as np
import pandas as pd
import sacrebleu
import time

assert len(external_df) == 1012

required_columns = [
    "fr",
    "original_translation",
    "finetuned_translation"
]

assert external_df[required_columns].notna().all().all()

references = external_df["fr"].tolist()
original = external_df["original_translation"].tolist()
finetuned = external_df["finetuned_translation"].tolist()

bleu_metric = sacrebleu.metrics.BLEU()
chrf_metric = sacrebleu.metrics.CHRF()

N_BOOTSTRAP = 1000
SEED = 42

rng = np.random.default_rng(SEED)

def calculate_difference(indices):
    refs = [references[i] for i in indices]
    old = [original[i] for i in indices]
    new = [finetuned[i] for i in indices]

    bleu_difference = (
        bleu_metric.corpus_score(new, [refs]).score
        - bleu_metric.corpus_score(old, [refs]).score
    )

    chrf_difference = (
        chrf_metric.corpus_score(new, [refs]).score
        - chrf_metric.corpus_score(old, [refs]).score
    )

    return bleu_difference, chrf_difference

all_indices = np.arange(len(external_df))

observed_bleu, observed_chrf = calculate_difference(
    all_indices
)

bootstrap_bleu = []
bootstrap_chrf = []

start_time = time.perf_counter()

for iteration in range(N_BOOTSTRAP):
    sampled_indices = rng.integers(
        0,
        len(external_df),
        size=len(external_df)
    )

    bleu_diff, chrf_diff = calculate_difference(
        sampled_indices
    )

    bootstrap_bleu.append(bleu_diff)
    bootstrap_chrf.append(chrf_diff)

    if (iteration + 1) % 250 == 0:
        print("Completed:", iteration + 1, "/", N_BOOTSTRAP)

rows = []

for metric_name, observed, samples in [
    ("BLEU", observed_bleu, bootstrap_bleu),
    ("chrF", observed_chrf, bootstrap_chrf)
]:
    lower, upper = np.percentile(
        samples,
        [2.5, 97.5]
    )

    rows.append({
        "Metric": metric_name,
        "Observed Improvement": round(observed, 3),
        "95% CI Lower": round(lower, 3),
        "95% CI Upper": round(upper, 3),
        "CI Excludes Zero": bool(lower > 0 or upper < 0)
    })

external_bootstrap_df = pd.DataFrame(rows)

print("\nFLORES-200 BOOTSTRAP ANALYSIS")
print("Sentences:", len(external_df))
print("Bootstrap repetitions:", N_BOOTSTRAP)
print("Random seed:", SEED)

display(external_bootstrap_df)

print(
    "\nAnalysis time:",
    round(time.perf_counter() - start_time, 2),
    "seconds"
)

external_bootstrap_df.to_csv(
    "external_flores_bootstrap.csv",
    index=False
)

print("\nBootstrap results saved.")


Completed: 250 / 1000
Completed: 500 / 1000
Completed: 750 / 1000
Completed: 1000 / 1000

FLORES-200 BOOTSTRAP ANALYSIS
Sentences: 1012
Bootstrap repetitions: 1000
Random seed: 42


,Metric,Observed Improvement,95% CI Lower,95% CI Upper,CI Excludes Zero
0,BLEU,0.351,-0.222,0.96,False
1,chrF,0.236,-0.092,0.62,False



Analysis time: 1405.89 seconds

Bootstrap results saved.


In [5]:

from google.colab import files

files.download("external_flores_metrics.csv")
files.download("external_flores_predictions.csv")
files.download("external_flores_bootstrap.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>